# Medical SLM — Stage A on Google Colab

This notebook runs the verified Stage A trainer. Use a GPU runtime. The dataset is extracted to Colab's local SSD; completed checkpoints are mirrored to Google Drive.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("BF16 supported:", torch.cuda.is_bf16_supported())

## Mount Drive and configure paths

Upload `stage-a-data.tar` to `MyDrive/medical-slm/` before running this cell.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

REPOSITORY_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
PROJECT_DIRECTORY = Path("/content/medical-slm")
DATA_ARCHIVE = Path("/content/drive/MyDrive/medical-slm/stage-a-data.tar")
DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/medical-slm-runs/stage_a_dev/checkpoints")

assert DATA_ARCHIVE.is_file(), f"Missing data archive: {DATA_ARCHIVE}"

## Create the isolated 100-update development profile

In [ ]:
import yaml

development = yaml.safe_load(Path("configs/training_stage_a_colab.yaml").read_text())
development.update({
    "output_directory": "/content/stage_a_dev",
    "checkpoint_backup_directory": str(DRIVE_CHECKPOINTS),
    "max_updates": 100,
    "validation_interval": 50,
    "checkpoint_interval": 50,
})
DEVELOPMENT_CONFIG = Path("/content/training_stage_a_colab_dev.yaml")
DEVELOPMENT_CONFIG.write_text(yaml.safe_dump(development, sort_keys=False))
print(DEVELOPMENT_CONFIG.read_text())

## Clone, install, and extract local training data

For a private repository, configure Git credentials through a Colab secret. Never place an access token directly in the notebook.

In [ ]:
import os
import shutil
import subprocess

if not PROJECT_DIRECTORY.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIRECTORY)], check=True)

os.chdir(PROJECT_DIRECTORY)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
subprocess.run(["tar", "-xf", str(DATA_ARCHIVE), "-C", str(PROJECT_DIRECTORY)], check=True)
print("Working directory:", Path.cwd())

In [ ]:
required = [
    Path("artifacts/tokenizer/tokenizer.json"),
    Path("datasets/tokenized/stage_a/dataset_manifest.json"),
    Path("datasets/tokenized/stage_a/train/metadata.json"),
    Path("datasets/tokenized/evaluation/dataset_manifest.json"),
    Path("datasets/tokenized/evaluation/validation/metadata.json"),
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f"Missing required files: {missing}"
shards = list(Path("datasets/tokenized/stage_a/train").glob("shard_*.bin"))
assert len(shards) == 115, f"Expected 115 Stage A shards, found {len(shards)}"
print("Dataset verified: 115 Stage A shards")

## Run the focused cloud-training tests

In [ ]:
subprocess.run([
    "python", "-m", "pytest",
    "tests/test_training_checkpoint.py",
    "tests/test_training_step.py",
    "tests/test_stage_a_trainer.py",
    "-q",
], check=True)

## Initial validation baseline

Run this once before development training. Random-initialization loss should be near `ln(16000) ≈ 9.68`.

In [ ]:
import yaml
from medical_slm.model import DecoderConfig
from medical_slm.training.config import load_stage_a_config
from medical_slm.training.trainer import StageATrainer

training_config = load_stage_a_config(DEVELOPMENT_CONFIG)
model_values = yaml.safe_load(Path("configs/model_stage_a.yaml").read_text())
baseline_trainer = StageATrainer(training_config, DecoderConfig.from_mapping(model_values))
baseline = baseline_trainer.evaluate()
baseline_trainer.metric_logger.close()
print({"loss": baseline.loss, "perplexity": baseline.perplexity, "tokens": baseline.tokens})

## Development run: train to update 50

The Colab profile mirrors the verified update-50 checkpoint to Drive automatically.

In [ ]:
subprocess.run([
    "python", "scripts/training/train_stage_a.py",
    "--config", str(DEVELOPMENT_CONFIG),
    "--max-updates", "50",
], check=True)

In [ ]:
assert (DRIVE_CHECKPOINTS / "latest.json").is_file()
print("Drive checkpoint backup:", DRIVE_CHECKPOINTS)
print((DRIVE_CHECKPOINTS / "latest.json").read_text())

## Restart/resume test

Restart the Colab session, then rerun the setup cells above. Run this cell to restore the mirrored checkpoint into local storage.

In [ ]:
LOCAL_CHECKPOINTS = Path("/content/stage_a_dev/checkpoints")
LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
subprocess.run([
    "rsync", "-a", f"{DRIVE_CHECKPOINTS}/", f"{LOCAL_CHECKPOINTS}/"
], check=True)
assert (LOCAL_CHECKPOINTS / "latest.json").is_file()
print("Restored:", (LOCAL_CHECKPOINTS / "latest.json").read_text())

In [ ]:
subprocess.run([
    "python", "scripts/training/train_stage_a.py",
    "--config", str(DEVELOPMENT_CONFIG),
    "--resume", "latest",
    "--max-updates", "100",
], check=True)

## Full Stage A training

Run only after the 50→100 resume test succeeds and its validation loss decreases. On a fresh runtime, restore Drive checkpoints first. Omit `--resume` only when starting from random initialization.

In [ ]:
# Fresh full run:
# subprocess.run(["python", "scripts/training/train_stage_a.py",
#                 "--config", "configs/training_stage_a_colab.yaml"], check=True)

# Resume an interrupted full run after restoring checkpoints from Drive:
# subprocess.run(["python", "scripts/training/train_stage_a.py",
#                 "--config", "configs/training_stage_a_colab.yaml",
#                 "--resume", "latest"], check=True)